# Quick Tour of `lsms_library`

This notebook demonstrates the core API using Tanzania's National Panel Survey (NPS).

We will:
1. List the countries available in the library
2. Instantiate a `Country` for Tanzania and inspect its waves and tables
3. Load `cluster_features` (geographic metadata)
4. Load `household_characteristics` with a market/region dimension
5. Load `food_expenditures` and `food_quantities` for demand analysis

## 0. Install the package

Install directly from the GitHub repository.  This also pulls in
dependencies (`pandas`, `dvc`, `pyarrow`, etc.).

In [ ]:
%pip install git+https://github.com/ligon/LSMS_Library.git -q

### Authentication

The underlying microdata is stored in a DVC remote.  If this is your
first time, run the cell below and enter the passphrase when prompted
(ask `ligon@berkeley.edu` for the passphrase).  Once credentials are
cached on disk this step is automatic.

In [ ]:
import lsms_library as ll

# Uncomment the line below if you haven't authenticated yet:
# ll.authenticate()

## 1. Which countries are available?

The `Feature` class discovers every country that declares a given table.
`household_roster` is the most universal, so it gives the broadest list.

In [ ]:
roster = ll.Feature('household_roster')
roster.countries

## 2. Instantiate a Country

`Country` is the primary interface.  It exposes the available survey
waves and the harmonized tables (the "data scheme").

In [ ]:
tz = ll.Country('Tanzania')

print(f"Waves  : {tz.waves}")
print(f"Tables : {tz.data_scheme}")

Tanzania has six waves spanning 2008--2021.  The first four share a
single multi-round data directory (`2008-15/`); the library handles
the mapping transparently.

Tables like `food_expenditures`, `food_prices`, and
`food_quantities` are *derived* at runtime from `food_acquired`.
Similarly, `household_characteristics` is derived from
`household_roster`.  No extra data files are needed.

## 3. Cluster features --- geographic context

`cluster_features` provides Region, District, and Rural/Urban
for each household-wave observation.

In [ ]:
cf = tz.cluster_features()

print(f"Shape  : {cf.shape}")
print(f"Index  : {cf.index.names}")
print(f"Columns: {cf.columns.tolist()}")
cf.head()

In [ ]:
# How many distinct regions per wave?
cf.groupby('t')['Region'].nunique()

## 4. Household characteristics

`household_characteristics` is a demographic summary automatically
derived from `household_roster`.

The `market='Region'` argument joins the Region from
`cluster_features` as an index level `m`.  This is the market
dimension used in CFE demand estimation --- each region is treated as
a separate price regime.

In [ ]:
hc = tz.household_characteristics(market='Region')

print(f"Shape  : {hc.shape}")
print(f"Index  : {hc.index.names}")
print(f"Columns: {hc.columns.tolist()[:8]}  ...")
hc.head()

In [ ]:
# Households per wave and market (region)
hc.groupby(['t', 'm']).size().unstack(fill_value=0)

## 5. Food expenditures

`food_expenditures` is derived from `food_acquired`.  With
`market='Region'` the returned DataFrame has index
`(i, t, m, j)` --- household, wave, market, food item --- ready
for demand estimation.

In [ ]:
x = tz.food_expenditures(market='Region')

print(f"Shape          : {x.shape}")
print(f"Index          : {x.index.names}")
print(f"Columns        : {x.columns.tolist()}")
print(f"Food items (j) : {x.index.get_level_values('j').nunique()}")
print(f"Waves          : {sorted(x.index.get_level_values('t').unique())}")
x.head(8)

In [ ]:
# Mean expenditure by wave
x.groupby('t')['Expenditure'].mean().round(1)

## 6. Food quantities

`food_quantities` converts survey-reported amounts into kilograms
using country-specific unit conversion factors.

In [ ]:
q = tz.food_quantities(market='Region')

print(f"Shape  : {q.shape}")
print(f"Index  : {q.index.names}")
print(f"Columns: {q.columns.tolist()}")
q.head(8)

In [ ]:
# Mean quantity (kg) by wave
q.groupby('t')['Quantity'].mean().round(2)

## Next steps

With `x` (expenditures) and `hc` (household characteristics) indexed
by `(i, t, m)`, you can estimate a CFE demand system in a few lines:

```python
import numpy as np
from cfe.regression import Regression

y = np.log(x['Expenditure'].replace(0, np.nan)).dropna()
r = Regression(y=y, d=hc, alltm=False)
r.get_beta()
```

See the [documentation](https://ligon.github.io/LSMS_Library/) and
`CONTRIBUTING.org` for details on adding new countries or features.